1. Read the data with json and csv file DirectoryLoader
2. Chunks
3. RecursiveTextSplitter
4. Embedding
5. FAISS Vector Database
6. LLM Chatbot with chat history, context and response

In [7]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader, JSONLoader, CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.chains import RetrievalQA
from sentence_transformers import SentenceTransformer

c:\Users\Srajan.Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!pip install langchain-google-genai

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your_key"

In [9]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings = OpenAIEmbeddings(model = "text-embedding-ada-002",
                                     openai_api_key = os.environ["OPENAI_API_KEY"])

In [10]:
from langchain_community.vectorstores import FAISS

In [11]:
jq_schema = ".[] | {instruction: .instruction, input: .input, output: .output}"
loader1= JSONLoader('D:/test/Medical/chatdoctor5k.json', jq_schema=jq_schema, text_content=False)

In [12]:
doc1 = loader1.load()
len(doc1)

5452

In [ ]:
loader2= CSVLoader('/test/Medical/format_dataset.csv')

In [14]:
doc2= loader2.load()
len(doc2)

796

In [15]:
doc3= doc1+doc2
len(doc3)

6248

In [53]:
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                                    chunk_overlap=200,
                                                    separators=["\n\n", "\n", ".", " "])

In [54]:
recursive_tokens = recursive_splitter.split_documents(doc3)

In [55]:
print(f"Number of tokens created from document: {len(recursive_tokens)}")

Number of tokens created from document: 6998


In [56]:
openai_embeddings = OpenAIEmbeddings(model = "text-embedding-ada-002",
                                     openai_api_key = os.environ["OPENAI_API_KEY"])
 

In [57]:
# GEMINI EMBEDDINGS

In [ ]:
import os
os.environ["GEMINI_API_KEY"]="your_key"

In [59]:
import google.genai as genai
client= genai.Client(api_key=os.environ['GEMINI_API_KEY'])

In [60]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [61]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [62]:
gemini_ai_embeddings = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001",
                                     google_api_key = os.environ["GEMINI_API_KEY"])

In [63]:
faiss_medical_store= FAISS.from_documents(documents=recursive_tokens,
                                  embedding=gemini_ai_embeddings)

GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0
Please retry in 18.875488606s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-embedding-1.0"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 100
}
, retry_delay {
  seconds: 18
}
]

In [65]:
faiss_medical_store= FAISS.from_documents(documents=recursive_tokens,
                                  embedding=openai_embeddings)

In [66]:
from langchain.memory import ConversationBufferMemory

In [67]:
memory = ConversationBufferMemory(memory_key="chat_history",
                                  return_messages=True, output_key='answer')

C:\Users\Srajan.Agrawal\AppData\Local\Temp\ipykernel_21276\1504241007.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history",


In [69]:
retriever = faiss_medical_store.as_retriever(search_type="mmr", search_kwargs={"k":3})

In [70]:
from langchain_core.prompts import PromptTemplate

In [71]:
prompt = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template="""You are an experienced medical assistant specializing in analyzing
doctor-patient conversations, clinical notes, diagnoses, symptoms,
treatments, medications, and healthcare-related information.

Use only the medical records, doctor-patient conversations, and clinical
documents provided in the context to answer the user's question.

When available, mention relevant details such as symptoms, diagnosis,
medications, treatment plans, test results, and clinical observations.

Provide clear, concise, and accurate responses based solely on the
provided context. Do not make assumptions, generate medical information
that is not present in the documents, or use external knowledge.

If the answer is not available in the provided documents, say
"I don't know based on the provided medical records.".
 
    Context:
    {context}
 
    Chat History:
    {chat_history}
 
    Question: {question}
 
    Answer:""")
 

In [72]:
from  langchain.chains import ConversationalRetrievalChain

In [73]:
from langchain_openai import ChatOpenAI
llm= ChatOpenAI(model_name="gpt-4.1-nano",
                temperature=0.5,
                top_p=0.75)

In [74]:
QA_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,      # to return the source documents along with the answer
    verbose=False,
    combine_docs_chain_kwargs={"prompt": prompt},
    output_key='answer'
)
 
print("Welcome to the Medical Database Query Chatbot!")

Welcome to the Medical Database Query Chatbot!


In [ ]:
def ask_question(question: str) -> dict:
    result = QA_chain.invoke({"question": question})
   
    print("=" * 70)
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print("-" * 70)
    print("Sources:")
    for i, doc in enumerate(result["source_documents"], start=1):
        meta   = doc.metadata
        source = meta.get("source", "Unknown").split("\\")[-1]  # filename only
        page   = meta.get("page", "?")
        print(f"  [{i}] {source}  |  Page {page}")k
        print(f"       {doc.page_content[:150].strip()}...")
    print("=" * 70)
 

In [80]:
ask_question("I am having irritation in my nose and sour throat?")

Q: I am having irritation in my nose and sour throat?
A: Based on your symptoms of nose irritation and sore throat, it is possible that you may have a nose disorder or an associated upper respiratory condition. The symptoms you described—nasal congestion, sore throat, and irritation—are common in conditions like sinusitis or other nasal disorders. 

However, I don't have enough specific information to determine a definitive diagnosis. If your symptoms persist or worsen, I recommend consulting a healthcare professional for an accurate assessment and appropriate treatment.
----------------------------------------------------------------------
Sources:
  [1] chatdoctor5k.json  |  Page ?
       {"instruction": "If you are a doctor, please answer the medical questions based on the patient's description.", "input": "Hi doctor, I've been experie...
  [2] D:/test/Medical/format_dataset.csv  |  Page ?
       idx: 745
disease: Tonsillitis
Symptom: ['Sore throat', 'Swollen or red tonsils', 'Abnor